In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
import torch

In [ ]:
# Load dataset
print("Loading tiny-codes dataset...")
ds = load_dataset("nampdn-ai/tiny-codes", split="train")

In [ ]:
# Load model and tokenizer
model_name = "/kaggle/input/qwen2.5/transformers/0.5b/1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

In [ ]:
# Tokenize the dataset
def tokenize_function(example):
    # If your dataset has a 'text' column
    return tokenizer(example["text"], truncation=True, max_length=512, padding="max_length")

tokenized_ds = ds.map(tokenize_function, batched=True, remove_columns=ds.column_names)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
)

In [ ]:
# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    tokenizer=tokenizer,
)

# 6. Start training
trainer.train()